# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue'] = df['qty'] * df['price']
print('Shape:', df.shape)
print('Revenue:', df['revenue'].sum())
df.head()

Shape: (400, 5)
Revenue: 8520.0


,vendor_id,category,qty,price,revenue
0,V-10,Drink,2,24.0,48.0
1,V-18,RainGear,1,12.0,12.0
2,V-18,Drink,3,4.5,13.5
3,V-10,Food,2,12.0,24.0
4,V-18,Drink,3,7.5,22.5


Report: The number of rows is 400, which is the same as the expected number of rows. Also, the total revenue is 8,520 dollars, which is between 8,000 dollars and 9,000 dollars.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [18]:
# TODO

# Revenue by category
revenue_by_category = df.groupby('category')['revenue'].sum()
# Sort highest to lowest
revenue_by_category = revenue_by_category.sort_values(ascending=False)

# Include the share of total as a percentage in the same table
revenue_share = revenue_by_category / revenue_by_category.sum() * 100
by_category = pd.concat([revenue_by_category, revenue_share], axis=1, keys=['Revenue', 'Revenue Share (%)'])

by_category

,Revenue,Revenue Share (%)
category,,
Food,4293.0,50.387324
Merch,1771.5,20.792254
Drink,1554.0,18.239437
RainGear,901.5,10.580986


Report: For the 'Food' category, the total revenue is \$4,293.0, which makes up ~50.39% of the total revenue. For the 'Merch' category, the total revenue is \$1771.5, which makes up ~20.79% of the total revenue. For the 'Drink' category, the total revenue is \$1554.0	, which makes up ~18.24% of the total revenue. For the 'RainGear' category, the total revenue is \$901.5	, which makes up ~10.58% of the total revenue.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO

# Vendor's average order revenue
vendor_avg_revenue = df.groupby('vendor_id')['revenue'].mean()
# Sort highest to lowest
vendor_avg_revenue = vendor_avg_revenue.sort_values(ascending=False)

# Vendor's order count
vendor_order_count = df.groupby('vendor_id').size()

q3_combined = pd.concat([vendor_avg_revenue, vendor_order_count], axis=1, keys=['Average Revenue', 'Order Count'])
q3_combined

,Average Revenue,Order Count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


Report: For vendor V-01, the average revenue is ~\$22.60 with an order count of 94. For vendor V-18, the average revenue is \$21.75 with an order count of 108. For vendor V-05, the average revenue is \$20.58 with an order count of 93. For vendor V-10, the average revenue is \$20.31 with an order count of 105.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO

print('Share of Revenue that Comes from Merch (%):', round(q2_combined.loc['Merch', 'Revenue Share (%)'], 1))

Share of Revenue that Comes from Merch (%): 20.8


Report: The share of the total revenue that comes from the 'Merch' category is ~20.8%.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [9]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')
joined.head()

print('Row Count Before Merge:', df.shape[0])
print('Row Count After Merge:', joined.shape[0])
print('Revenue Total Before Merge:', df['revenue'].sum())
print('Revenue Total After Merge:', joined['revenue'].sum())
print('Unmatched Vendor:', joined[joined['vendor_name'].isna()]['vendor_id'].unique())

Row Count Before Merge: 400
Row Count After Merge: 400
Revenue Total Before Merge: 8520.0
Revenue Total After Merge: 8520.0
Unmatched Vendor: ['V-18']


**The unmatched vendor, and what I did about it:** The unmatched vender is V-18. To merge df with vendor_names, I did a many_to_one left join on vendor_id. Because V-18 does not have a matching entry in vendor_names, the vendor_name column is filled with NaN for those rows.

To validate the merge, I compared the row count and the total revenue before and after the merge. The values stayed the same, showing that the left join did not add or remove any rows and that the total revenue was preserved.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [26]:
# TODO

# Make a pivot table
# Add row/column totals
pivot_table = pd.pivot_table(joined, values='revenue', index='vendor_name', columns='category', aggfunc='sum', margins=True, margins_name='Total')
pivot_table

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Total,972.0,3274.5,1263.0,661.5,6171.0


Report: This pivot table shows the 'joined' data frame as a report. Each row

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [21]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
# assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

_your answer here_